In [10]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score


def find_file(candidates):
  for path in candidates:
    if os.path.exists(path):
      return path
  raise FileNotFoundError(f"None of these files were found: {candidates}")


clf_path = find_file([
    "data/classification_escalation.csv",
    "classification_escalation.csv",
    "classification_escalation[1].csv",
])

print("Using classification file:", clf_path)

clf_df = pd.read_csv(clf_path)

print("\nClassification shape:", clf_df.shape)
print("Classification missing:\n", clf_df.isna().sum())
print("\nEscalated value counts:\n", clf_df["escalated"].value_counts())
display(clf_df.head())

Using classification file: classification_escalation.csv

Classification shape: (900, 6)
Classification missing:
 complexity_score           0
team_experience_years     31
priority                  18
channel                    0
customer_tenure_months    22
escalated                  0
dtype: int64

Escalated value counts:
 escalated
0    714
1    186
Name: count, dtype: int64


,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,9,7.7,low,web,21.6,0
1,2,6.1,high,web,21.8,0
2,1,2.5,medium,phone,28.8,0
3,7,5.3,low,email,52.5,0
4,4,7.1,medium,email,14.7,0


In [5]:
target_col = "escalated"
drop_cols = [c for c in ["ticket_id", target_col] if c in clf_df.columns]
X_clf = clf_df.drop(columns=drop_cols)
y_clf = clf_df[target_col]

# Split BEFORE preprocessing and hold out the test set
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

num_cols_c = X_train_c.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_c = X_train_c.select_dtypes(exclude=[np.number]).columns.tolist()

prep_c = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
        ]),
        num_cols_c,
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]),
        cat_cols_c,
    ),
])

print("Train shape:", X_train_c.shape, "| Test shape:", X_test_c.shape)
print("Numeric cols:", num_cols_c)
print("Categorical cols:", cat_cols_c)

Train shape: (720, 5) | Test shape: (180, 5)
Numeric cols: ['complexity_score', 'team_experience_years', 'customer_tenure_months']
Categorical cols: ['priority', 'channel']


In [7]:
cv_c = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_c = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "KNN": KNeighborsClassifier(),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
}

cv_rows_c = []
for name, clf in models_c.items():
  pipe = Pipeline([("prep", prep_c), ("model", clf)])
  scores = cross_validate(
      pipe,
      X_train_c,
      y_train_c,
      cv=cv_c,
      scoring="f1",
      return_train_score=True,
  )
  cv_rows_c.append({
      "Model": name,
      "Train F1 Mean": round(scores["train_score"].mean(), 4),
      "CV F1 Mean": round(scores["test_score"].mean(), 4),
      "CV F1 Std": round(scores["test_score"].std(), 4),
  })

cv_table_c = pd.DataFrame(cv_rows_c)
display(cv_table_c)

,Model,Train F1 Mean,CV F1 Mean,CV F1 Std
0,LogisticRegression,0.2697,0.2281,0.0453
1,KNN,0.4973,0.3362,0.0604
2,DecisionTree,1.0000,0.3487,0.0557


In [8]:
grid_lr = GridSearchCV(
    Pipeline([
        ("prep", prep_c),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]),
    param_grid={"model__C": [0.01, 0.1, 1.0, 10.0, 100.0]},
    cv=cv_c,
    scoring="f1",
)
grid_lr.fit(X_train_c, y_train_c)

grid_dt = GridSearchCV(
    Pipeline([("prep", prep_c), ("model", DecisionTreeClassifier(random_state=42))]),
    param_grid={
        "model__max_depth": [2, 3, 4, 5, 6, 8, None],
        "model__min_samples_leaf": [1, 5, 10, 20],
    },
    cv=cv_c,
    scoring="f1",
)
grid_dt.fit(X_train_c, y_train_c)

print("Best LogisticRegression:", grid_lr.best_params_, "| CV F1:", round(grid_lr.best_score_, 4))
print("Best DecisionTree:    ", grid_dt.best_params_, "| CV F1:", round(grid_dt.best_score_, 4))

best_grid_c = grid_lr if grid_lr.best_score_ >= grid_dt.best_score_ else grid_dt
print("Selected best estimator:", type(best_grid_c.best_estimator_.named_steps["model"]).__name__, best_grid_c.best_params_)

Best LogisticRegression: {'model__C': 1.0} | CV F1: 0.2281
Best DecisionTree:     {'model__max_depth': None, 'model__min_samples_leaf': 1} | CV F1: 0.3487
Selected best estimator: DecisionTreeClassifier {'model__max_depth': None, 'model__min_samples_leaf': 1}


In [9]:
y_pred_c = best_grid_c.predict(X_test_c)

print("Test F1:", round(f1_score(y_test_c, y_pred_c), 4))
print("Confusion Matrix:\n", confusion_matrix(y_test_c, y_pred_c))
print("\nClassification Report:\n", classification_report(y_test_c, y_pred_c, digits=4))

Test F1: 0.3333
Confusion Matrix:
 [[120  23]
 [ 25  12]]

Classification Report:
               precision    recall  f1-score   support

           0     0.8276    0.8392    0.8333       143
           1     0.3429    0.3243    0.3333        37

    accuracy                         0.7333       180
   macro avg     0.5852    0.5817    0.5833       180
weighted avg     0.7279    0.7333    0.7306       180



## 3. Experiment A — Classification Results, Error Analysis & Justification

### Validation Strategy, Candidate Models & Search Space
* **Validation strategy:** **5-fold Stratified Cross-Validation (`StratifiedKFold`)** on the `720`-row training set so each fold preserves the escalation rate (~20.69% in train, ~20.56% in test).
* **Candidate models & tuning search space:** Compared `LogisticRegression`, `KNeighborsClassifier` (`KNN`), and `DecisionTreeClassifier` using 5-fold CV `F1` (`scoring="f1"`). Then tuned `LogisticRegression` (`C` in `[0.01, 0.1, 1.0, 10.0, 100.0]`) and `DecisionTreeClassifier` (`max_depth` in `[2, 3, 4, 5, 6, 8, None]` and `min_samples_leaf` in `[1, 5, 10, 20]`) with `GridSearchCV`.

### 5-Fold Stratified CV Results Table
 **Model**          | **Train F1 Mean** | **CV F1 Mean** | **CV F1 Std** |
 LogisticRegression | `0.2697`          | `0.2281`       | `0.0453`      |
 KNN                | `0.4973`          | `0.3362`       | `0.0604`      |
 **DecisionTree     | `1.0000`          | `0.3487`       | `0.0557`      |

### Hyperparameter Tuning & One-Time Test Evaluation
* **Best tuned `LogisticRegression`:** `{'model__C': 1.0}` with **CV F1 = `0.2281`**.
* **Best tuned `DecisionTreeClassifier`:** `{'model__max_depth': None, 'model__min_samples_leaf': 1}` with **CV F1 = `0.3487`**.
* **Selected best estimator:** `DecisionTreeClassifier` (`max_depth=None`, `min_samples_leaf=1`), chosen because its CV F1 (`0.3487`) was higher than `LogisticRegression` (`0.2281`) and `KNN` (`0.3362`).
* **Held-out test set evaluation (`180` tickets: `143` class `0`, `37` class `1`):**
  * **Test F1:** `0.3333` (Class `1` precision = `0.3429`, recall = `0.3243`, F1 = `0.3333`; Class `0` F1 = `0.8333`; overall accuracy = `0.7333`).
  * **Confusion Matrix (`[[TN, FP], [FN, TP]]`):**
    * True Negatives (`TN`): `120`
    * False Positives (`FP`): `23`
    * False Negatives (`FN`): `25`
    * True Positives (`TP`): `12`

### False Positives vs. False Negatives
* **False Positives (`23`):** Non-escalated tickets predicted as escalated (`1`). These trigger unnecessary escalation flags or extra review effort, adding operational overhead without causing an unmonitored escalation.
* **False Negatives (`25` out of `37` escalated tickets):** Escalated tickets predicted as non-escalated (`0`). Missing `25` of `37` escalations (~67.6% of actual escalations) is more critical in support operations because high-urgency or complex tickets may not get timely escalation or attention.

### Final Model Justification & Limitations
* **Justification:** Following the automated CV F1 selection rule, `DecisionTreeClassifier` (`max_depth=None`, `min_samples_leaf=1`) achieved the highest 5-fold CV F1 (`0.3487`) on this split, outperforming `KNN` (`0.3362`) and `LogisticRegression` (`0.2281`).
* **Limitations:** The selected unrestricted decision tree clearly **overfits** the training data (`Train F1 Mean = 1.0000` vs. `CV F1 Mean = 0.3487` and `Test F1 = 0.3333`), memorizing training branches rather than generalizing strongly to unseen tickets. Overall recall (`0.3243`) and precision (`0.3429`) on escalated tickets remain modest, indicating that stronger regularization/pruning, ensemble models, or threshold adjustment would be needed in practice.